# 🔄 CNN-Autoencoder with Data Augmentation for Brain MRI Anomaly Detection

## 📋 Overview

This notebook implements a **CNN-Autoencoder with heavy data augmentation** to handle geometric variations.

### Model Architecture
- **Type**: Convolutional Autoencoder (same as baseline)
- **Input**: 128×128 grayscale MRI slice
- **Encoder**: 4 Conv2D layers with MaxPooling
- **Decoder**: 4 ConvTranspose2D layers for upsampling
- **Latent**: 256-dimensional bottleneck

### 🔄 Data Augmentation Strategy
To handle rotation invariance **manually**, we apply:
- **Random Rotation**: ±15° (simulates arbitrary tumor orientations)
- **Random Horizontal Flip**: 50% probability
- **Random Vertical Flip**: 50% probability
- **Brightness/Contrast Jitter**: ±10% (handles imaging variations)

### Research Question
> "Can data augmentation achieve rotation invariance comparable to ECNN's built-in equivariance?"

### Expected Performance
- **AUROC**: ~0.78-0.84 (better than baseline, but not as good as ECNN)
- **Training Time**: ~45-60 min (50 epochs, augmentation adds overhead)
- **Key Insight**: Augmentation helps but is computationally expensive vs. ECNN's built-in property

---

## 1️⃣ Setup and Environment Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print("✅ Google Drive mounted successfully!")

In [ ]:
# Keep Colab session alive
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button");
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click();
   }
   btn = document.querySelector('#ok');
   if (btn != null){
     console.log("Click connect button");
     btn.click();
   }
 }
 setInterval(ClickConnect, 60000)
'''))

print("✅ Keep-alive script activated!")

## ⚡ Turbo Data Loading (Local Disk)

**Why this matters**: Loading 33k+ files from Google Drive is SLOW (~30 min). Instead:
1. **Check** if zip files exist (created once)
2. **Extract** to local runtime disk (~2 min)
3. **Train** with blazing fast I/O (10-20x speedup)

**Note**: The zip files were created during CNN-AE training.

In [ ]:
import os
from google.colab import drive

# Check for the zips
base = "/content/drive/MyDrive/symAD-ECNN/data"
zips = [f"{base}/train_fast.zip", f"{base}/val_fast.zip", f"{base}/test_fast.zip"]

missing = [f for f in zips if not os.path.exists(f)]

if len(missing) == 0:
    print("✅ GOOD NEWS: Zip files found! Proceeding to extraction...")
else:
    print("⚠️ WARNING: Zip files missing. Please run the CNN-AE notebook first to create them.")
    print(f"   Missing: {missing}")

In [ ]:
# ==========================================
# ⚡ TURBO LOADER (Unzip to Local)
# ==========================================
import zipfile
import os
import shutil

BASE_DIR = "/content/drive/MyDrive/symAD-ECNN/data"
LOCAL_DIR = "/content/local_data"

ZIPS = {
    "train": f"{BASE_DIR}/train_fast.zip",
    "val":   f"{BASE_DIR}/val_fast.zip",
    "test":  f"{BASE_DIR}/test_fast.zip"
}

print("🚀 Extracting to Local Disk...")

for name, zip_path in ZIPS.items():
    target_dir = f"{LOCAL_DIR}/{name}"
    # Clean up old run if exists
    if os.path.exists(target_dir): 
        shutil.rmtree(target_dir)
    os.makedirs(target_dir, exist_ok=True)

    if os.path.exists(zip_path):
        print(f"   📂 Unzipping {name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
    else:
        print(f"   ❌ ERROR: {zip_path} not found!")

print("\n✅ Data Ready! Local folders created.")

In [ ]:
# Install required packages
!pip install pytorch-msssim -q

print("✅ All packages installed!")

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
from pytorch_msssim import SSIM
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc, confusion_matrix
from glob import glob
import os
import time
from tqdm import tqdm
import json

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True

print("✅ All libraries imported successfully!")

In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Define paths
BASE_PATH = "/content/drive/MyDrive/symAD-ECNN"

# ⚡ DATA PATHS (Point to LOCAL DISK for speed) ⚡
IXI_TRAIN_PATH = "/content/local_data/train"
IXI_VAL_PATH   = "/content/local_data/val"
BRATS_PATH     = "/content/local_data/test"

# Model and results paths (Keep on Drive!)
MODEL_PATH = f"{BASE_PATH}/models/saved_models/cnn_ae_augmented"
RESULTS_PATH = f"{BASE_PATH}/results/cnn_ae_augmented"

# Create directories
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("📁 Paths configured:")
print(f"   ⚡ Data (Local): {IXI_TRAIN_PATH}")
print(f"   💾 Models (Drive): {MODEL_PATH}")
print(f"   📊 Results (Drive): {RESULTS_PATH}")

## 2️⃣ Data Loading with Augmentation

### 🔄 Augmentation Strategy
We apply aggressive augmentations **only during training**:
- Random rotation (±15°)
- Random horizontal/vertical flips
- Brightness/contrast jitter

**Validation and test sets remain unchanged** for fair comparison.

In [ ]:
# Dataset class WITH augmentation
class MRIDatasetAugmented(Dataset):
    """
    Dataset with data augmentation for training.
    
    Augmentations:
    - Random rotation: ±15°
    - Random horizontal flip: 50%
    - Random vertical flip: 50%
    - Brightness jitter: ±10%
    - Contrast jitter: ±10%
    """
    def __init__(self, file_list, augment=True):
        self.files = file_list
        self.augment = augment
        
        # Define augmentation transforms
        if augment:
            self.transform = transforms.Compose([
                transforms.RandomRotation(degrees=15, fill=0),  # ±15° rotation
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.1, contrast=0.1),
            ])
        else:
            self.transform = None

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        try:
            # Load .npy file
            img = np.load(self.files[idx])
            
            # Convert to tensor first (C, H, W)
            img_tensor = torch.from_numpy(img).float().unsqueeze(0)  # (1, 128, 128)
            
            # Apply augmentation if training
            if self.augment and self.transform is not None:
                img_tensor = self.transform(img_tensor)
            
            # Return (input, target) pair - same for autoencoder
            return img_tensor, img_tensor
        except Exception as e:
            # Fallback for corrupted files
            print(f"Error loading {self.files[idx]}: {e}")
            return torch.zeros((1, 128, 128)), torch.zeros((1, 128, 128))

print("✅ Augmented Dataset class defined!")

In [ ]:
# Load file paths
print("📂 Loading file paths from local disk...")

train_files = sorted(glob(f"{IXI_TRAIN_PATH}/*.npy"))
val_files = sorted(glob(f"{IXI_VAL_PATH}/*.npy"))
brats_files = sorted(glob(f"{BRATS_PATH}/*.npy"))

# Validation checks
if len(train_files) == 0:
    raise ValueError(f"❌ No training files found in {IXI_TRAIN_PATH}!")

if len(val_files) == 0:
    raise ValueError(f"❌ No validation files found in {IXI_VAL_PATH}!")

if len(brats_files) == 0:
    print(f"⚠️ WARNING: No BraTS test files found in {BRATS_PATH}")

print(f"✅ Found {len(train_files):,} IXI training slices")
print(f"✅ Found {len(val_files):,} IXI validation slices")
print(f"✅ Found {len(brats_files):,} BraTS test slices (tumors)")

In [ ]:
# Create datasets and dataloaders
train_dataset = MRIDatasetAugmented(train_files, augment=True)   # WITH augmentation
val_dataset = MRIDatasetAugmented(val_files, augment=False)      # NO augmentation
test_dataset = MRIDatasetAugmented(brats_files, augment=False)   # NO augmentation

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✅ DataLoaders created!")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)} (WITH augmentation)")
print(f"   Val batches: {len(val_loader)} (NO augmentation)")
print(f"   Test batches: {len(test_loader)} (NO augmentation)")

## 3️⃣ Visualize Augmentation Examples

Let's see what the augmentations look like!

In [ ]:
# Visualize augmentation examples
print("🔍 Visualizing augmentation examples...")

# Get one sample
original_img = np.load(train_files[100])
original_tensor = torch.from_numpy(original_img).float().unsqueeze(0)

# Apply augmentation multiple times
augment_transform = transforms.Compose([
    transforms.RandomRotation(degrees=15, fill=0),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
])

# Generate augmented versions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Original
axes[0, 0].imshow(original_tensor[0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Original', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Augmented versions
for idx in range(1, 8):
    row = idx // 4
    col = idx % 4
    augmented = augment_transform(original_tensor.clone())
    axes[row, col].imshow(augmented[0].numpy(), cmap='gray', vmin=0, vmax=1)
    axes[row, col].set_title(f'Augmented #{idx}', fontsize=12, fontweight='bold')
    axes[row, col].axis('off')

plt.suptitle('Data Augmentation Examples: Random Rotations, Flips, Brightness/Contrast', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Augmentation visualization saved!")
print("\n" + "="*60)
print("📊 AUGMENTATION DETAILS:")
print("="*60)
print("🔄 Rotation: ±15° (handles arbitrary tumor orientations)")
print("↔️ Horizontal Flip: 50% probability")
print("↕️ Vertical Flip: 50% probability")
print("💡 Brightness: ±10% (imaging variations)")
print("🌓 Contrast: ±10% (scanner differences)")
print("="*60)

## 4️⃣ CNN-Autoencoder Model (Same as Baseline)

In [ ]:
class CNNAutoencoder(nn.Module):
    """
    Convolutional Autoencoder (SAME architecture as baseline)
    
    Encoder: 128×128 -> 64×64 -> 32×32 -> 16×16 -> 8×8
    Decoder: 8×8 -> 16×16 -> 32×32 -> 64×64 -> 128×128
    """

    def __init__(self, latent_dim=256):
        super(CNNAutoencoder, self).__init__()

        # Encoder: Progressive downsampling
        self.encoder = nn.Sequential(
            # 128×128×1 -> 128×128×32
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 64×64

            # 64×64×32 -> 64×64×64
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 32×32

            # 32×32×64 -> 32×32×128
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 16×16

            # 16×16×128 -> 16×16×256
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2)  # 8×8
        )

        # Latent space
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 256, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 256)

        # Decoder: Progressive upsampling
        self.decoder = nn.Sequential(
            # 8×8×256 -> 16×16×256
            nn.ConvTranspose2d(256, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            # 16×16×256 -> 32×32×128
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # 32×32×128 -> 64×64×64
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            # 64×64×64 -> 128×128×32
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),

            # 128×128×32 -> 128×128×1
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Encode
        features = self.encoder(x)

        # Latent
        batch_size = features.size(0)
        flat = self.flatten(features)
        z = self.fc_encode(flat)

        # Decode
        decoded_flat = self.fc_decode(z)
        decoded_features = decoded_flat.view(batch_size, 256, 8, 8)
        x_recon = self.decoder(decoded_features)

        return x_recon

    def get_latent(self, x):
        features = self.encoder(x)
        flat = self.flatten(features)
        return self.fc_encode(flat)

model = CNNAutoencoder().to(device)
total_params = sum(p.numel() for p in model.parameters())

print("🖼️ CNN-Autoencoder Created (with augmentation training)!")
print(f"   Total parameters: {total_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024**2:.2f} MB")
print(f"   Architecture: SAME as baseline")
print(f"   Difference: Training with heavy augmentation")

## 5️⃣ Loss Function and Optimizer

In [ ]:
class CombinedLoss(nn.Module):
    """
    Combined MSE + SSIM Loss

    Loss = α * MSE + (1-α) * (1 - SSIM)
    """
    def __init__(self, alpha=0.84):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        self.ssim = SSIM(data_range=1.0, size_average=True, channel=1, win_size=11)

    def forward(self, output, target):
        mse_loss = self.mse(output, target)
        ssim_loss = 1 - self.ssim(output, target)
        combined = self.alpha * mse_loss + (1 - self.alpha) * ssim_loss
        return combined

criterion = CombinedLoss(alpha=0.84)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
scaler = GradScaler()

print("✅ Combined Loss Function created!")
print(f"   MSE weight (α): 0.84")
print(f"   SSIM weight (1-α): 0.16")
print(f"   Mixed precision: Enabled")

## 6️⃣ Robust Training Loop (Inline & Optimized)

In [ ]:
# ==========================================
# 🔄 ROBUST TRAINING LOOP (Inline & Optimized)
# ==========================================
from tqdm.notebook import tqdm

NUM_EPOCHS = 50
KEEP_LAST_N_CHECKPOINTS = 3

# Check for existing checkpoints
checkpoints = sorted(glob(f'{MODEL_PATH}/cnn_aug_epoch*.pth'))
RESUME = len(checkpoints) > 0

if RESUME:
    latest = checkpoints[-1]
    print(f"✅ Found checkpoint: {os.path.basename(latest)}")
    checkpoint = torch.load(latest, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch']
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))
    best_epoch = checkpoint.get('best_epoch', 0)
    print(f"   Resuming from epoch {start_epoch}")
    print(f"   Current LR: {optimizer.param_groups[0]['lr']:.2e}")
else:
    start_epoch = 0
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_epoch = 0
    print("📝 Starting fresh training")

print(f"\n🚀 Training CNN-AE (Augmented): Epochs {start_epoch + 1} to {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}, Device: {device}")
print(f"   Augmentation: ON (Train only)")
print("-" * 60)

start_time = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()
    
    # --- 1. TRAINING (With Augmentation) ---
    model.train()
    epoch_train_loss = 0
    # Loop over the Augmented Training Set
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)
    
    for imgs, targets in loop:
        imgs, targets = imgs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        
        # Robust Autocast
        with torch.amp.autocast('cuda'):
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- 2. VALIDATION (No Augmentation) ---
    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            epoch_val_loss += loss.item()
            
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Update Scheduler
    scheduler.step(avg_val_loss)

    # --- 3. LOGGING & SAVING ---
    epoch_time = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}] | Train: {avg_train_loss:.6f} | Val: {avg_val_loss:.6f} | LR: {current_lr:.2e} | {epoch_time/60:.1f}min")

    # Save Best Model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        torch.save({

            'epoch': epoch + 1,print(f"   Best Epoch: {best_epoch}, Best Val Loss: {best_val_loss:.6f}")

            'model_state_dict': model.state_dict(),print(f"   Total Time: {total_time/3600:.2f} hours")

            'optimizer_state_dict': optimizer.state_dict(),print(f"\n🎉 Training Complete!")

            'scheduler_state_dict': scheduler.state_dict(),total_time = time.time() - start_time

            'val_loss': avg_val_loss,

            'best_val_loss': best_val_loss,    print("-" * 60)

            'best_epoch': best_epoch

        }, f'{MODEL_PATH}/cnn_aug_best.pth')            print(f"   🗑️ Deleted old checkpoint: {os.path.basename(old_ckpt)}")

        print(f"   💾 Best model saved!")            os.remove(old_ckpt)

        for old_ckpt in checkpoints[:-KEEP_LAST_N_CHECKPOINTS]:

    # Save Regular Checkpoint    if len(checkpoints) > KEEP_LAST_N_CHECKPOINTS:

    torch.save({    checkpoints = sorted(glob(f'{MODEL_PATH}/cnn_aug_epoch*.pth'))

        'epoch': epoch + 1,    # Cleanup Old Checkpoints

        'model_state_dict': model.state_dict(),    

        'optimizer_state_dict': optimizer.state_dict(),    }, f'{MODEL_PATH}/cnn_aug_epoch{epoch+1}.pth')

        'scheduler_state_dict': scheduler.state_dict(),        'best_epoch': best_epoch

        'train_losses': train_losses,        'best_val_loss': best_val_loss,
        'val_losses': val_losses,

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train', linewidth=2)
plt.plot(val_losses, label='Val', linewidth=2)
plt.axvline(x=best_epoch-1, color='r', linestyle='--', label=f'Best ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN-AE with Augmentation: Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train', linewidth=2)
plt.plot(val_losses, label='Val', linewidth=2)
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss (log)')
plt.title('CNN-AE with Augmentation: Training (Log Scale)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/cnn_aug_training_curves.png', dpi=150)
plt.show()

## 8️⃣ Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(f'{MODEL_PATH}/cnn_aug_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✅ Best model loaded (Epoch {checkpoint['epoch']}, Val Loss: {checkpoint['val_loss']:.6f})")

# Calculate reconstruction errors
def calculate_errors(model, dataloader, device):
    model.eval()
    errors = []
    with torch.no_grad():
        for data, _ in tqdm(dataloader, desc='Computing errors'):
            data = data.to(device)
            recon = model(data)
            mse = nn.functional.mse_loss(recon, data, reduction='none').view(data.size(0), -1).mean(dim=1)
            errors.extend(mse.cpu().numpy())
    return np.array(errors)

normal_errors = calculate_errors(model, val_loader, device)
anomaly_errors = calculate_errors(model, test_loader, device)

# Metrics
y_true = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(anomaly_errors))])
y_scores = np.concatenate([normal_errors, anomaly_errors])
auroc = roc_auc_score(y_true, y_scores)
precision, recall, _ = precision_recall_curve(y_true, y_scores)
auprc = auc(recall, precision)

print(f"\n📈 CNN-AE with Augmentation Performance:")
print(f"   AUROC: {auroc:.4f}")
print(f"   AUPRC: {auprc:.4f}")
print(f"   Normal errors: {normal_errors.mean():.6f} ± {normal_errors.std():.6f}")
print(f"   Anomaly errors: {anomaly_errors.mean():.6f} ± {anomaly_errors.std():.6f}")

# Plot
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(normal_errors, bins=50, alpha=0.7, label='Normal', density=True, color='blue')
plt.hist(anomaly_errors, bins=50, alpha=0.7, label='Anomaly', density=True, color='red')
plt.xlabel('Reconstruction Error')
plt.ylabel('Density')
plt.legend()
plt.title('CNN-AE with Augmentation: Error Distribution')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(y_true, y_scores)
plt.plot(fpr, tpr, linewidth=2, label=f'CNN-AE Aug (AUROC={auroc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.title('CNN-AE with Augmentation: ROC Curve')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/cnn_aug_evaluation.png', dpi=150)
plt.show()

# Save results
results = {
    'model': 'CNN-Autoencoder-Augmented',
    'auroc': float(auroc),
    'auprc': float(auprc),
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'normal_error_mean': float(normal_errors.mean()),
    'anomaly_error_mean': float(anomaly_errors.mean()),
    'total_params': total_params,
    'training_time_hours': total_time / 3600,
    'augmentation': {
        'rotation': '±15°',
        'horizontal_flip': '50%',
        'vertical_flip': '50%',
        'brightness': '±10%',
        'contrast': '±10%'
    }
}

with open(f'{RESULTS_PATH}/cnn_aug_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("\n✅ CNN-AE with augmentation evaluation complete!")

## 9️⃣ Visualization: Reconstruction Maps

In [ ]:
def visualize_reconstructions(model, dataloader, device, num_samples=5, title_prefix=""):
    """Visualize original images, reconstructions, and error maps"""
    model.eval()
    
    data, _ = next(iter(dataloader))
    data = data[:num_samples].to(device)
    
    with torch.no_grad():
        recon = model(data)
    
    error_maps = torch.abs(data - recon)
    
    data_np = data.cpu().numpy()
    recon_np = recon.cpu().numpy()
    error_np = error_maps.cpu().numpy()
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        axes[i, 0].imshow(data_np[i, 0], cmap='gray', vmin=0, vmax=1)
        axes[i, 0].set_title(f'{title_prefix} Original', fontsize=12, fontweight='bold')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(recon_np[i, 0], cmap='gray', vmin=0, vmax=1)
        mse_val = np.mean((data_np[i, 0] - recon_np[i, 0])**2)
        axes[i, 1].set_title(f'Reconstruction (MSE={mse_val:.6f})', fontsize=12, fontweight='bold')
        axes[i, 1].axis('off')
        
        im = axes[i, 2].imshow(error_np[i, 0], cmap='hot', vmin=0, vmax=error_np[i, 0].max())
        axes[i, 2].set_title(f'Error Map (Max={error_np[i, 0].max():.4f})', fontsize=12, fontweight='bold')
        axes[i, 2].axis('off')
        plt.colorbar(im, ax=axes[i, 2], fraction=0.046)
    
    plt.tight_layout()
    return fig

# Visualize Normal Cases
print("🔍 Visualizing Normal Cases...")
fig_normal = visualize_reconstructions(model, val_loader, device, num_samples=5, title_prefix="Normal")
plt.savefig(f'{RESULTS_PATH}/cnn_aug_reconstruction_normal.png', dpi=150, bbox_inches='tight')
plt.show()

# Visualize Anomaly Cases
print("🔍 Visualizing Anomaly Cases...")
fig_anomaly = visualize_reconstructions(model, test_loader, device, num_samples=5, title_prefix="Anomaly")
plt.savefig(f'{RESULTS_PATH}/cnn_aug_reconstruction_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("📊 KEY INSIGHT:")
print("="*60)
print("🔄 Data augmentation helps the model handle rotations")
print("   BUT it's computationally expensive during training")
print("⚡ ECNN achieves this BUILT-IN without augmentation overhead")
print("="*60)

## 🔟 Comparative Metrics (Confusion Matrix & Threshold Analysis)

This section provides **apples-to-apples comparison** with the baseline CNN-AE to answer:
> **"Did augmentation reduce the False Positives?"**

In [ ]:
# ==========================================
# 📊 COMPARATIVE METRICS (Confusion Matrix)
# ==========================================
print("🎯 Calculating Optimal Threshold & Confusion Matrix...")

# 1. Calculate Optimal Threshold (Youden's J statistic)
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"✅ Optimal Threshold: {optimal_threshold:.6f}")

# 2. Confusion Matrix
predictions = (y_scores > optimal_threshold).astype(int)
cm = confusion_matrix(y_true, predictions)
tn, fp, fn, tp = cm.ravel()

accuracy = (tp + tn) / len(y_true)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "="*60)
print("📈 AUGMENTED MODEL RESULTS (Vs Baseline)")
print("="*60)
print(f"Threshold:    {optimal_threshold:.6f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"Specificity:  {specificity:.4f}  <-- COMPARE THIS TO BASELINE (~0.58)")
print(f"Precision:    {precision:.4f}")
print(f"F1-Score:     {f1:.4f}")
print("="*60)
print(f"False Positives: {fp:,}  <-- DID THIS GO DOWN?")
print(f"False Negatives: {fn:,}")
print("="*60)

# 3. Visualize Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Normal', 'Anomaly'], 
            yticklabels=['Normal', 'Anomaly'])
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_title(f'Confusion Matrix\n(Threshold={optimal_threshold:.6f})', 
                 fontsize=13, fontweight='bold')

# Metrics Bar Chart
metrics_names = ['Recall\n(TPR)', 'Specificity\n(TNR)', 'Precision', 'F1-Score', 'Accuracy']
metrics_values = [recall, specificity, precision, f1, accuracy]
colors_bar = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6']

bars = axes[1].bar(metrics_names, metrics_values, color=colors_bar, alpha=0.8)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Performance Metrics (CNN-AE Augmented)', fontsize=13, fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, metrics_values):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/cnn_aug_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# 4. Save Extended Results
results['metrics'] = {
    'threshold': float(optimal_threshold),
    'recall': float(recall),
    'specificity': float(specificity),
    'precision': float(precision),
    'f1': float(f1),
    'accuracy': float(accuracy),
    'fp': int(fp),
    'fn': int(fn),
    'tp': int(tp),
    'tn': int(tn)
}

with open(f'{RESULTS_PATH}/cnn_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print(f"\n✅ Extended results saved to: {RESULTS_PATH}/cnn_results.json")
print("📊 Use these metrics to compare with baseline CNN-AE (Notebook 02)")